# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, overview, and analyze the [FAIR^2 clinicopathological dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSet `@id`s and their respective field `@id`s
print("Available record sets:\n----------------------")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    # List fields in this recordset
    if 'field' in rs:
        # Field may be a list or a single dict
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for f in fields:
            # f might be a dict or a str id
            if isinstance(f, dict):
                print(f"    - @id: {f.get('@id', f)} ; name: {f.get('name', 'N/A')}")
            else:
                print(f"    - @id: {f}")
    print("")
if not record_sets:
    print("No explicit record sets found via `dataset.record_sets`.")

# If no recordSets are found via the property, try via the raw metadata
if not record_sets and hasattr(metadata, 'to_json'):
    md_json = metadata.to_json()
    record_sets = md_json.get('recordSet', [])
    if record_sets:
        print("Found record sets via metadata JSON:")
        for rs in record_sets:
            print(f"- RecordSet @id: {rs.get('@id', rs)}")
    else:
        print("No record sets found in metadata JSON.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# We'll obtain the RecordSet @ids from the dataset. If not available, fall back to default.
from pprint import pprint

def extract_all_record_sets(dataset):
    # Attempt to access via property
    try:
        return [rs['@id'] for rs in dataset.record_sets]
    except Exception:
        try:
            # Fallback to JSON
            md = dataset.metadata
            if hasattr(md, 'to_json'):
                js = md.to_json()
                rs = js.get('recordSet', [])
                if rs and isinstance(rs[0], dict):
                    return [x['@id'] for x in rs]
                elif rs:
                    return rs
        except Exception:
            pass
    return []

record_sets_ids = extract_all_record_sets(dataset)
if not record_sets_ids:
    print("No record sets explicitly declared. Will attempt extraction with guess.")
    # Use default known record set id for this dataset
    # For this dataset, the record set is likely at '@id': 'https://api.app.sen.science/frontiers/7862866/629c16ec-37ee-4556-a351-d5164116c2dd#recordSet-1'
    record_sets_ids = [
        'https://api.app.sen.science/frontiers/7862866/629c16ec-37ee-4556-a351-d5164116c2dd#recordSet-1'
    ]

print("Record set @ids to extract:")
pprint(record_sets_ids)

dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

# For demonstration, use the first available dataframe
main_record_set_id = record_sets_ids[0]
print(f"\nAvailable columns in RecordSet {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# Find a numeric field to use (e.g., 'Age' or a field containing numbers)
df = dataframes[main_record_set_id]

# List columns containing likely numeric data
numeric_candidates = [col for col in df.columns if any(s in col.lower() for s in ['age', 'interval', 'year', 'count'])]
print(f"Numeric field candidates: {numeric_candidates}")
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Use first likely numeric
else:
    # Fallback: use the first column containing numeric data
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.empty else None

print(f"Using numeric field for EDA: {numeric_field}")

# We'll filter on sensible threshold
if numeric_field and numeric_field in df:
    # Coerce non-numeric if needed
    df_num = pd.to_numeric(df[numeric_field], errors='coerce')
    # Use 60 as age threshold, or 5 as interval threshold, else 10
    threshold = 60 if 'age' in numeric_field.lower() else (5 if 'interval' in numeric_field.lower() else 10)
    filtered_df = df[df_num > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold} (n={filtered_df.shape[0]}):")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (df_num[filtered_df.index] - df_num.mean()) / df_num.std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a key column (e.g., 'Sex', 'MSI status', 'Anatomical Site', or any string column)
    group_field_candidates = [col for col in df.columns if col.lower() in ['sex', 'msi-status', 'msi_status', 'anatomical-site', 'anatomical_site', 'tumor-location', 'tumor_location'] or 'site' in col.lower() or 'sex' in col.lower() or 'status' in col.lower()]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No obvious group field found.")
else:
    print("No numeric field available to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if numeric_field and numeric_field in df:
    plt.figure(figsize=(7,4))
    pd.to_numeric(df[numeric_field], errors='coerce').hist(bins=20, alpha=0.8)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field}')
    plt.tight_layout()
    plt.show()

# Boxplot by group_field
if 'group_field' in locals() and group_field and group_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df.dropna(subset=[group_field, numeric_field]))
    plt.title(f'{numeric_field} by {group_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load tabular data defined via a Croissant schema, explore available record sets and fields (all referenced by their `@id`s), extract records to a DataFrame, perform basic preprocessing and EDA, and create initial visualizations.

**Key observations:**
- The dataset is highly structured and includes rich clinicopathological variables for 77 cancer survivors with second primary colorectal cancer.
- Numeric and categorical fields (e.g., age, MSI status) allow for standard analytic workflows in Pandas.
- The use of Croissant and `mlcroissant` makes it possible to programmatically load both metadata and data using unique `@id` references for record sets and fields, supporting reproducible FAIR data analysis.

Further data exploration could include advanced statistical modeling, deeper stratification by anatomical or molecular subtype, or exporting results in FAIR-compliant formats.